# Sprint 4: Dashboarding the E-Commerce Dataset in Power BI & BI Tools

### Project Mission
You are a **Business Intelligence (BI) Analyst & Analytics Consultant**. Having normalized and analyzed the e-commerce dataset in SQL (Sprint 3), executive stakeholders now need an intuitive, interactive dashboard to monitor product performance, pricing benchmarks, and customer satisfaction.

---

### Executive Dashboard Wireframe & Layout

```text
+----------------------------------------------------------------------------------+
|                         EXECUTIVE E-COMMERCE DASHBOARD                           |
+-----------------------+--------------------------+-------------------------------+
|    [ KPI CARD 1 ]     |      [ KPI CARD 2 ]      |        [ KPI CARD 3 ]         |
|     Average Price     |      Average Rating      |        Total Products         |
|        $116.89        |           4.02           |              103              |
+-----------------------+--------------------------+-------------------------------+
|   [ FILTERS / SLICERS ]:  Category: [ All | Electronics | Home-Goods | Sports ]  |
|                           Price Range Slider: [$0 ----------o----------- $350]   |
+------------------------------------------+---------------------------------------+
|      CATEGORY & BRAND PRICE SPREAD       |       PRICE vs RATING MATRIX          |
|               (Bar Chart)                |            (Scatter Plot)             |
|  Electronics:  ============ $162.40      |  Rating                               |
|  Home-Goods:   =======      $95.10       |   5.0|    *       *    *              |
|  Sports:       ======       $89.50       |   4.0| *      *   *        *          |
|                                          |   3.0|    *       *                   |
|                                          |      +------------------------ Price  |
+------------------------------------------+---------------------------------------+
```


## Q1: What is the "Philosophy of Dashboarding" and how do we define the 3 Core Business Questions?

### Concept Pointers:
- **The Golden Rule**: Never open Power BI or Tableau without first defining the **Audience Persona** and **3 Core Business Questions**.
- **Target Audience Persona**: **Category Manager (Electronics)**.
- **The 3 Essential Questions**:
  1. *Which product categories and brands are priced significantly above the catalog average?*
  2. *Where is customer satisfaction (rating) weakest relative to price?*
  3. *How does catalog composition shift when filtered across price tiers ($0-$100, $100-$250, $250+)?*

## Q2: How do we prepare and inspect our reporting datasets for Power BI?

### Concept Pointers:
- In Sprint 3, data was stored in 3 normalized relational tables. Flat denormalized tables optimize reporting performance in Power BI.
- We inspect `data/products_for_powerbi.csv` and `data/review_summary_for_powerbi.csv`.

In [1]:
import pandas as pd

# Load input tables
products_df = pd.read_csv("data/products_for_powerbi.csv")
reviews_summary_df = pd.read_csv("data/review_summary_for_powerbi.csv")

# Clean missing review counts to avoid visualization errors
products_df["review_count"] = products_df["review_count"].fillna(0).astype(int)

print(f"Products Table:       {len(products_df)} rows")
print(f"Review Summary Table: {len(reviews_summary_df)} rows")
products_df.head(3)


Products Table:       103 rows
Review Summary Table: 103 rows


,product_id,name,brand,category,price,rating,review_count
0,1,Maple Row Electric Kettle,Maple Row,home-goods,23.29,3.7,1269
1,2,Lumen Air Purifier,Lumen,home-goods,186.52,2.9,2408
2,4,Maple Row Wall Clock,Maple Row,home-goods,127.00,4.7,859


In [2]:
# Merge tables on product_id to view full reporting dataset
dashboard_df = products_df.merge(reviews_summary_df, on="product_id", how="left")
print("Combined Reporting Dataset Preview:")
dashboard_df[["product_id", "name", "category", "price", "rating", "actual_review_count", "avg_review_rating"]].head(3)


Combined Reporting Dataset Preview:


,product_id,name,category,price,rating,actual_review_count,avg_review_rating
0,1,Maple Row Electric Kettle,home-goods,23.29,3.7,9,4.00
1,2,Lumen Air Purifier,home-goods,186.52,2.9,5,2.80
2,4,Maple Row Wall Clock,home-goods,127.00,4.7,3,4.33


## Q3: How do we compute executive KPI card metrics?

### Concept Pointers:
- **KPI Card 1: Total Products** -> Count of catalog listings.
- **KPI Card 2: Average Price** -> Overall catalog average price.
- **KPI Card 3: Average Rating** -> Flat average rating vs **Weighted Average Rating** (weighted by review volume).

In [3]:
# Step 1: Compute Total Products & Average Price
total_products = len(products_df)
avg_price = products_df["price"].mean()

print(f"Total Products: {total_products}")
print(f"Average Price:  ${avg_price:.2f}")


Total Products: 103
Average Price:  $192.59


In [4]:
# Step 2: Compute Flat vs Weighted Average Rating
avg_rating = products_df["rating"].mean()
total_reviews = products_df["review_count"].sum()
weighted_avg_rating = (products_df["rating"] * products_df["review_count"]).sum() / total_reviews

print(f"Flat Average Rating:     {avg_rating:.2f} / 5.0")
print(f"Weighted Average Rating: {weighted_avg_rating:.2f} / 5.0")


Flat Average Rating:     3.81 / 5.0
Weighted Average Rating: 3.68 / 5.0


In [5]:
# Step 3: Format Executive KPI Summary
print("==========================================")
print("         EXECUTIVE KPI SUMMARY            ")
print("==========================================")
print(f"  Total Active Products:   {total_products}")
print(f"  Catalog Average Price:   ${avg_price:.2f}")
print(f"  Flat Average Rating:     {avg_rating:.2f} / 5.0")
print(f"  Weighted Average Rating: {weighted_avg_rating:.2f} / 5.0")
print("==========================================")


         EXECUTIVE KPI SUMMARY            
  Total Active Products:   103
  Catalog Average Price:   $192.59
  Flat Average Rating:     3.81 / 5.0
  Weighted Average Rating: 3.68 / 5.0


## Q4: How do we build an interactive prototype of the dashboard using Plotly?

### Concept Pointers:
- Visual 1: Category Price Comparison Bar Chart (Answering Question 1).
- Visual 2: Price vs Rating Scatter Matrix (Answering Question 2).
- Interactive Slicer: Price threshold filter simulation (Answering Question 3).

In [6]:
import plotly.express as px

# Visual 1: Average Price by Category
cat_avg_df = products_df.groupby("category")["price"].mean().reset_index()
cat_avg_df = cat_avg_df.sort_values(by="price", ascending=False)

fig_bar = px.bar(
    cat_avg_df,
    x="category",
    y="price",
    color="category",
    title="Average Price by Category (Benchmark Analysis)",
    text_auto=".2f",
    labels={"price": "Average Price ($)", "category": "Category"}
)
fig_bar.update_layout(showlegend=False, template="plotly_white")
fig_bar.show()


In [7]:
# Visual 2: Price vs Rating Scatter Matrix
plot_df = products_df.copy()
plot_df["display_size"] = plot_df["review_count"].clip(lower=10)

fig_scatter = px.scatter(
    plot_df,
    x="price",
    y="rating",
    color="category",
    size="display_size",
    hover_data=["name", "brand", "review_count"],
    title="Customer Rating vs Product Price (Bubble Size = Review Count)",
    labels={"price": "Price ($)", "rating": "Rating (1-5)"}
)
fig_scatter.update_layout(template="plotly_white")
fig_scatter.show()


In [8]:
# Visual 3: Slicer filtering simulation function
def filter_dashboard_by_price(max_price=150.0):
    """Simulate Power BI dynamic slicer filtering."""
    filtered = products_df[products_df["price"] <= max_price]
    print(f"--- Slicer Applied: Price <= ${max_price:.2f} ---")
    print(f"  Products in View:   {len(filtered)} / {len(products_df)}")
    print(f"  Filtered Avg Price: ${filtered['price'].mean():.2f}")
    print(f"  Filtered Avg Rating: {filtered['rating'].mean():.2f}")

filter_dashboard_by_price(max_price=150.0)


--- Slicer Applied: Price <= $150.00 ---
  Products in View:   52 / 103
  Filtered Avg Price: $83.07
  Filtered Avg Rating: 3.81


## Q5: How do DAX Filter Contexts work in Power BI?

### Concept Pointers:
- **Row Context**: Evaluates row-by-row.
- **Filter Context**: Set of active filters from slicers, page filters, and chart selections.
- **`CALCULATE`**: Modifies the filter context.
- **`ALLEXCEPT`**: Clears all filters except on the specified column.

### DAX Formulas Reference:

#### 1. Average Price
```dax
Average Price = AVERAGE ( products[price] )
```

#### 2. Total Products
```dax
Total Products = COUNTROWS ( products )
```

#### 3. Above-Category-Average Flag
```dax
Above-Category-Average Flag =
VAR CategoryAvg =
    CALCULATE (
        AVERAGE ( products[price] ),
        ALLEXCEPT ( products, products[category] )
    )
RETURN
    IF ( products[price] > CategoryAvg, "Above Avg", "At/Below Avg" )
```

#### 4. Weighted Average Rating
```dax
Weighted Avg Rating =
AVERAGEX ( products, products[rating] * products[review_count] ) /
SUMX ( products, products[review_count] )
```

#### 5. Filtered Dynamic Price
```dax
Filtered Avg Price =
CALCULATE (
    AVERAGE ( products[price] ),
    products[price] >= MIN ( price_range_slicer[value] ),
    products[price] <= MAX ( price_range_slicer[value] )
)
```


## Q6: How do we open and present the solution in Power BI Desktop (`.pbix`)?

### Concept Pointers:
1. **Native Solution File**: Open `Sprint4_Ecommerce_Dashboard.pbix` in Microsoft Power BI Desktop.
2. **Data Model View**: Inspect the `1:1` relationship between `products` and `review_summary` on `product_id`.
3. **Report View**: Interact with the KPI cards, category comparison chart, scatter plot, and slicers.
4. **Web Browser Alternative**: Open `dashboard_design_preview.html` in Chrome or Edge for a zero-install interactive mockup.

## Summary & Key Takeaways

- **Philosophy of Dashboarding**: Started with the Category Manager persona and 3 clear questions.
- **KPI Cards**: Established executive context with Catalog Count, Average Price, and Weighted Rating.
- **Comparative Analytics**: Visualized category pricing spreads and evaluated satisfaction against price points.
- **DAX Mastery**: Explained row context vs filter context, `CALCULATE`, and `ALLEXCEPT`.
- **Full 4-Sprint Pipeline Complete**: From Raw System Logs (Sprint 1) -> Web Scraping & EDA (Sprint 2) -> Relational SQL Normalization (Sprint 3) -> Executive BI Dashboarding (Sprint 4)!